# Análise do Stack Tecnológico de Dados no Cepel

Este notebook realiza a importação dos dados do questionário, faz limpeza básica, organiza as tabelas no banco SQLite `cepel_data.db` e executa agregações (como `groupby` e `melt`) para analisar o uso e a prioridade das tecnologias de dados no Cepel.

**Principais etapas:**
- **Carregamento de dados brutos** a partir de arquivos `.csv` exportados do formulário.
- **Criação/conexão com o banco SQLite** (`cepel_data.db`) para armazenar as diferentes partes do questionário.
- **Consultas SQL** para recuperar as tabelas consolidadas.
- **Transformações com `pandas`** (por exemplo `groupby`, `agg`, `melt`) para gerar visões analíticas.

Use este notebook como referência para reproduzir a análise completa ou para adaptar o fluxo a novos conjuntos de dados do mesmo formulário.

In [ ]:
# Exemplo simples de agrupamento (GROUP BY) usando pandas
# OBS: este bloco é apenas um exemplo ilustrativo, não faz parte do fluxo principal da análise.

import pandas as pd

# Carrega um arquivo CSV de exemplo com colunas: departamento, tecnologia e uso
df = pd.read_csv("New Text Document.csv")

# Conta quantas entradas existem para cada combinação de departamento, tecnologia e uso
resultado = df.groupby(["departamento", "tecnologia", "uso"]).size().reset_index(name="quantidade")

In [ ]:
# Exporta o resultado do GROUP BY para CSV
# Esta célula considera que o DataFrame `resultado` já foi criado em células anteriores.

import pandas as pd

# Salva o DataFrame em um arquivo CSV simples (sem índice)
resultado.to_csv('testd.csv', index=False)

print("Arquivo salvo como 'dados_melted.csv'")

Arquivo salvo como 'dados_melted.csv'


In [52]:
# Agrupa o DataFrame por departamento e tipo de uso,
# contando quantos registros existem em cada combinação.
resultado = df.groupby(["departamento", "uso"]).agg(
    quantidade=("name", "count")
).reset_index()

In [ ]:
# Conexão com o banco SQLite e listagem das tabelas disponíveis
# O arquivo `cepel_data.db` deve ter sido criado previamente a partir dos arquivos do questionário.

#conn.close()

import sqlite3
conn = sqlite3.connect("cepel_data.db")

import pandas as pd

# Lista o nome de todas as tabelas presentes no banco SQLite
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)

print(tables)

# Carrega um recorte da tabela de identificação dos respondentes
table1 = pd.read_sql_query('''SELECT * FROM Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_0_Identificação1_21''', conn)
table1.head(3)

# Carrega a tabela referente à parte 3 do questionário (Interface / DevOps)
table4 = pd.read_sql_query('''SELECT * FROM Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_3_Interface_DevOps1_18''', conn)

,ID,START_TIME,COMPLETION_TIME,EMAIL,NAME,LAST_MODIFIED_TIME,FAVOR_INSERIR_SEU_NOME_COMPLETO,FAVOR_INSERIR_SEU_EMAIL_CEPELBR,EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO,FAVOR_APONTAR_SUA_FAIXA_ETARIA,FAVOR_APONTAR_O_SEU_TEMPO_DE_CASA_NO_CEPEL,FAVOR_APONTAR_SEU_MAIOR_NIVEL_DE_FORMACAO_EDUCACIONAL,HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS,HA_QUANTO_TEMPO_VOCE_JA_TRABALHA_COM_DADOS_GOVERNANCA_ENGENHARIA_CIENCIA_OU_ANALISE_DE_DADOS
0,6,2026-03-03 10:50:23,2026-03-03 10:51:24,0335009@cepel.br,Pedro Cesar de Oliveira,None,Pedro César Miguel de Oliveira,pedrocesar@cepel.br,DT/DTS - Depto de Transição Energética e Suste...,25-34 anos,6-10 anos,Especialização,5-10 anos,3-5 anos
1,7,2026-03-03 10:51:09,2026-03-03 10:52:20,0360841@cepel.br,Nério José Fernandes dos Santos,None,Nério José Fernandes dos Santos,nerio.santos@cepel.br,DT/DTS - Depto de Transição Energética e Suste...,25-34 anos,3-5 anos,Graduação,3-5 anos,3-5 anos
2,8,2026-03-03 10:57:39,2026-03-03 10:58:59,0360710@cepel.br,Marcelo Alberto Guimarães,None,Marcelo Alberto Guimarães,marcelo.guimaraes@cepel.br,"DT/DPC - Depto de Planejamento Energético, Com...",35-44 anos,1-2 anos,Mestrado / MBA,5-10 anos,5-10 anos


In [41]:
# Configuração das colunas e transformação `melt` para a parte 3 (Interface / DevOps)
# A ideia é transformar as várias colunas binárias em linhas, facilitando análises por tecnologia.

table4 = pd.read_sql_query('''SELECT * FROM Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_3_Interface_DevOps1_18''', conn)

'''
# Definindo as colunas separadamente para melhor legibilidade
colunas_fixas = ['NAME']

colunas_para_melt = [
    # Sistemas Operacionais
    'WINDOWS_NT_SERVER', 'WINDOWS_1011', 'LINUX_DEBIAN_UBUNTU_MINT_KALI', 
    'LINUX_ARCH_MANJARO_ENDEAVOUROS', 'LINUX_RED_HAT_FEDORA_CENTOS_ROCKY', 
    'LINUX_SUSE', 'LINUX_ALPINE', 'MACOS', 'CHROME_OS', 'ANDROID', 'IOS',
    
    # Virtualização
    'SSH', 'RDP_WINDOWS', 'VNC_MULTIPLATAFORMA', 'WSL_WINDOWS_SUBSYSTEM_FOR_LINUX',
    'VMWARE', 'VIRTUALBOX', 'PROXMOX', 'DOCKER', 'PODMAN', 'SINGULARITY',
    'DOCKERCOMPOSE', 'DOCKERSWARM', 'KUBERNETES',
    
    # Linguagens de Programação
    'SQL', 'PYTHON', 'R', 'SCALA', 'JAVA', 'JAVASCRIPT', 'TYPESCRIPT',
    'SHELL_SCRIPT', 'PHP', 'RUBY', 'GO', 'RUST', 'C_E_C', 'C', 'JULIA',
    'FORTRAN', 'MATLAB', 'LABVIEW2',
    
    # IDEs e Ferramentas de Desenvolvimento
    'MS_VISUAL_STUDIO', 'VS_CODE', 'CURSOR', 'ANTIGRAVITY', 'ECLIPSE',
    'INTELLIJ_IDEA', 'NETBEAMS', 'PYCHARM', 'SUBLIME', 'VIM', 'EMACS',
    'LABVIEW', 'CODER_CDE', 'JUPYTER_LABNOTEBOOK', 'GITHUB_CODESPACES',
    'KAGGLE_NOTEBOOKS', 'GOOGLE_COLABORATORY', 'BACKSTAGE_IDP', 'OPSLEVEL',
    'COMPASS', 'PORT',
    
    # Controle de Versão
    'CVS_CONCURRENT_VERSION_SYSTEM', 'SVN_APACHE_SUBVERSION', 'MERCURIAL',
    'GIT_GITHUB', 'GIT_GITLAB', 'GIT_BITBUCKET', 'GIT_LFS_LARGE_FILE_STORAGE',
    'DVC_DATA_VERSION_CONTROL', 'LAKEFS',
    
    # Configuração de Recursos
    'TERRAFORM', 'OPENTOFU', 'PULUMI', 'CROSSPLANE', 'ANSIBLE', 'PUPPET',
    'CHEF', 'SALT', 'NORNIR', 'VAGRANT',
    
    # CI/CD
    'JENKINS', 'TRAVIS_CI', 'CIRCLE_CI', 'GITHUB_ACTIONS', 'GITLAB_CICD',
    'TEAMCITY', 'BITBUCKET_PIEPELINES', 'FLUXCD', 'ARGOCD', 'OCTOPUSDEPLOY',
    
    # Metodologias de Gestão
    'PMBOK', 'SWEBOK', 'DMBOK', 'KANBAN', 'SCRUM', 'EXTREME_PROGRAMMING_XP', 'SAFE',
    
    # Boas Práticas de Desenvolvimento
    'DEVOPS', 'DEVSECOPS_OWASP_TOP_10', 'DATAOPS', 'MLOPS', 'CLEAN_ARCHITECTURE_E_CLEAN_CODE',
    'DDD_DOMAINDRIVEN_DESIGN', 'DESIGN_PATTERNS', 'BDD_BEHAVIORDRIVEN_DESIGN',
    'TDD_TESTDRIVEN_DEVELOPMENT', 'CRISPDM', 'PRINCIPIOS_SOLID', 'PRINCIPIOS_KISS',
    'PRINCIPIOS_YAGNI', 'REFACTORING', 'THE_TWELVEFACTOR_APP',
    
    # Ferramentas de Apoio Metodológico
    'SCOPI', 'BSC_DESIGNER', 'TRELLO', 'PLANNER', 'CLICKUP', 'JIRA', 'LINEAR',
    'GLPI', 'MANTIS', 'BUGZILLA',
    
    # Documentação
    'MS_SHAREPOINT', 'CONFLUENCE', 'NOTION', 'MONDAYCOM', 'MIRO', 'MS_WHITEBOARD',
    'SLACK', 'MS_TEAMS', 'DOXYGEN', 'DRAWIO', 'PLANTUML', 'SWAGGER', 'CKAN',
    'RESTRUCTUREDTEXT_REST', 'SPHINX', 'MKDOCS', 'READ_THE_DOCS', 'KEEP_A_CHANGELOG',
    'SCRIBE', 'OBS_STUDIO',
    
    # Campos de texto livre e prioridades
    'OUTRO6', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_SISTEMA_OPERACIONAL_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO',
    'QUAIS_DOS_SEGUINTES_SISTEMAS_OPERACIONAIS_LHE_SAO_PRIORITARIOS',
    'OUTRO7', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_MODELO_DE_VIRTUALIZACAO_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO',
    'QUAIS_DOS_SEGUINTES_MODELOS_DE_VIRTUALIZACAO_OU_DE_ACESSO_VIRTUAL_A_RECURSOS_LHE_SAO_PRIORITARIOS',
    'OUTRA', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRA_LINGUAGEM_DE_PROGRAMACAO_VOLTADA_A_DADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO',
    'QUAIS_DAS_SEGUINTES_LINGUAGENS_DE_PROGRAMACAO_VOLTADAS_A_DADOS_LHE_SAO_PRIORITARIAS',
    'OUTRO10', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_DESENVOLVIMENTO_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO',
    'QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_DESENVOLVIMENTO_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS',
    'OUTRO2', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_CONTROLE_DE_VERSAO_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO',
    'QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_CONTROLE_DE_VERSAO_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS',
    'OUTRO5', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_CONFIGURACAO_DE_RECURSOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO',
    'QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_CONFIGURACAO_DE_RECURSOS_LHE_SAO_PRIORITARIOS',
    'OUTRO3', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_CICD_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO',
    'QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_CICD_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS',
    'OUTRO', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_COMPENDIO_DE_BOAS_PRATICAS_PARA_GESTAO_DE_PROJETOS_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO',
    'QUAIS_DOS_SEGUINTES_COMPENDIOS_DE_BOAS_PRATICAS_PARA_GESTAO_DE_PROJETOS_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS',
    'OUTRO9', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_COMPENDIO_DE_BOAS_PRATICAS_EM_DESENVOLVIMENTO_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO',
    'QUAIS_DOS_SEGUINTES_COMPENDIOS_DE_BOAS_PRATICAS_EM_DESENVOLVIMENTO_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS',
    'OUTRO4', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_APOIO_METODOLOGICO_AO_DESENVOLVIMENTO_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO',
    'QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_APOIO_METODOLOGICO_AO_DESENVOLVIMENTO_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS',
    'OUTRO8', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_DOCUMENTACAO_DE_PROJETOSSOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO',
    'QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_DOCUMENTACAO_DE_PROJETOSSOFTWAREDADOS_LHE_SAO_PRIORITARIOS'
]

# Aplicando o melt (transforma colunas em linhas para facilitar a análise por tecnologia)
df_long = table4.melt(
    id_vars=colunas_fixas,
    value_vars=colunas_para_melt,
    var_name='tecnologia',
    value_name='resposta'
)
'''

"\n# Definindo as colunas separadamente para melhor legibilidade\ncolunas_fixas = ['NAME']\n\ncolunas_para_melt = [\n    # Sistemas Operacionais\n    'WINDOWS_NT_SERVER', 'WINDOWS_1011', 'LINUX_DEBIAN_UBUNTU_MINT_KALI', \n    'LINUX_ARCH_MANJARO_ENDEAVOUROS', 'LINUX_RED_HAT_FEDORA_CENTOS_ROCKY', \n    'LINUX_SUSE', 'LINUX_ALPINE', 'MACOS', 'CHROME_OS', 'ANDROID', 'IOS',\n\n    # Virtualização\n    'SSH', 'RDP_WINDOWS', 'VNC_MULTIPLATAFORMA', 'WSL_WINDOWS_SUBSYSTEM_FOR_LINUX',\n    'VMWARE', 'VIRTUALBOX', 'PROXMOX', 'DOCKER', 'PODMAN', 'SINGULARITY',\n    'DOCKERCOMPOSE', 'DOCKERSWARM', 'KUBERNETES',\n\n    # Linguagens de Programação\n    'SQL', 'PYTHON', 'R', 'SCALA', 'JAVA', 'JAVASCRIPT', 'TYPESCRIPT',\n    'SHELL_SCRIPT', 'PHP', 'RUBY', 'GO', 'RUST', 'C_E_C', 'C', 'JULIA',\n    'FORTRAN', 'MATLAB', 'LABVIEW2',\n\n    # IDEs e Ferramentas de Desenvolvimento\n    'MS_VISUAL_STUDIO', 'VS_CODE', 'CURSOR', 'ANTIGRAVITY', 'ECLIPSE',\n    'INTELLIJ_IDEA', 'NETBEAMS', 'PYCHARM', 'SUBLIME', 

In [5]:
list(table4.columns)

['ID',
 'START_TIME',
 'COMPLETION_TIME',
 'EMAIL',
 'NAME',
 'LAST_MODIFIED_TIME',
 'WINDOWS_NT_SERVER',
 'WINDOWS_1011',
 'LINUX_DEBIAN_UBUNTU_MINT_KALI',
 'LINUX_ARCH_MANJARO_ENDEAVOUROS',
 'LINUX_RED_HAT_FEDORA_CENTOS_ROCKY',
 'LINUX_SUSE',
 'LINUX_ALPINE',
 'MACOS',
 'CHROME_OS',
 'ANDROID',
 'IOS',
 'OUTRO6',
 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_SISTEMA_OPERACIONAL_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO',
 'QUAIS_DOS_SEGUINTES_SISTEMAS_OPERACIONAIS_LHE_SAO_PRIORITARIOS',
 'SSH',
 'RDP_WINDOWS',
 'VNC_MULTIPLATAFORMA',
 'WSL_WINDOWS_SUBSYSTEM_FOR_LINUX',
 'VMWARE',
 'VIRTUALBOX',
 'PROXMOX',
 'DOCKER',
 'PODMAN',
 'SINGULARITY',
 'DOCKERCOMPOSE',
 'DOCKERSWARM',
 'KUBERNETES',
 'OUTRO7',
 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_MODELO_DE_VIRTUALIZACAO_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO',
 'QUAIS_DOS_SEGUINTES_MODELOS_DE_VIRTUALIZACAO_OU_DE_ACESSO_VIRTUAL_A_RECURSOS_LHE_SAO_PRIORITARIOS',
 'SQL',
 'PYTHON',
 'R',
 'SCALA',
 

In [ ]:
'''
var_id_vars = ['ID', 'EMAIL', 'NAME']

var_value_vars = [
    'WINDOWS_NT_SERVER',
    'WINDOWS_1011',
    'LINUX_DEBIAN_UBUNTU_MINT_KALI',
    'LINUX_ARCH_MANJARO_ENDEAVOUROS',
    'LINUX_RED_HAT_FEDORA_CENTOS_ROCKY',
    'LINUX_SUSE',
    'LINUX_ALPINE',
    'MACOS',
    'CHROME_OS',
    'ANDROID',
    'IOS',
    'OUTRO6'
]

var_var_name = 'SISTEMAS_OPERACIONAIS'
var_value_name = 'RESPOSTA'

datafram = table4


def main():
    df_long = datafram.melt(
        id_vars=var_id_vars,
        value_vars=var_value_vars,
        var_name=var_var_name,
        value_name=var_value_name
    )

    df_long[var_var_name] = df_long[var_var_name].replace({'OUTRO6': 'OUTRO'})

    return df_long
'''


In [42]:
id_vars = [
    'ID',
    'EMAIL',
    'NAME'
]

datafram = table4

In [ ]:
so_cols = [
    'WINDOWS_NT_SERVER',
    'WINDOWS_1011',
    'LINUX_DEBIAN_UBUNTU_MINT_KALI',
    'LINUX_ARCH_MANJARO_ENDEAVOUROS',
    'LINUX_RED_HAT_FEDORA_CENTOS_ROCKY',
    'LINUX_SUSE',
    'LINUX_ALPINE',
    'MACOS',
    'CHROME_OS',
    'ANDROID',
    'IOS',
    'OUTRO6'
]

df_so = (
    datafram
    .melt(
        id_vars=id_vars,
        value_vars=so_cols,
        var_name="SISTEMAS_OPERACIONAIS",
        value_name="RESPOSTA_SISTEMAS_OPERACIONAIS"
    )
)

df_so["RESPOSTA_SISTEMAS_OPERACIONAIS"] = df_so["RESPOSTA_SISTEMAS_OPERACIONAIS"].replace({'OUTRO6': 'OUTRO'})






so_cols_ = ['SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_SISTEMA_OPERACIONAL_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO']

df_so_ = (
    datafram
    .melt(
        id_vars=id_vars,
        value_vars=so_cols_,
        var_name="SISTEMAS_OPERACIONAIS_APONTADOS",
        value_name="RESPOSTA_SISTEMAS_OPERACIONAIS_APONTADOS"
    )
)






so_cols__ = ['QUAIS_DOS_SEGUINTES_SISTEMAS_OPERACIONAIS_LHE_SAO_PRIORITARIOS'
]

df_so__ = (
    datafram
    .melt(
        id_vars=id_vars,
        value_vars=so_cols__,
        var_name="SISTEMAS_PRIORITARIOS",
        value_name="RESPOSTA_SISTEMAS_PRIORITARIOS"
    )
)





df_final = (
    df_so
    .merge(df_so_, on=["ID","EMAIL","NAME"])
    .merge(df_so__, on=["ID","EMAIL","NAME"])
)


'''
df_final = (
    df_so
    .merge(df_so_.drop(columns=["ID","EMAIL"]), on="NAME")
    .merge(df_so__.drop(columns=["ID","EMAIL"]), on="NAME")
)


df_final = df_so.merge(
    df_so_,
    on="NAME",
    suffixes=("_so", "_prioridade")
)
'''


In [ ]:
df_final

In [ ]:
# Survey Long Format / Tidy Data

# Criado por:
# Hadley Wickham
# É o padrão usado em:
# Data Science
# BI
# Machine Learning
# Survey Analytics

In [ ]:
so_cols = [
    'WINDOWS_NT_SERVER',
    'WINDOWS_1011',
    'LINUX_DEBIAN_UBUNTU_MINT_KALI',
    'LINUX_ARCH_MANJARO_ENDEAVOUROS',
    'LINUX_RED_HAT_FEDORA_CENTOS_ROCKY',
    'LINUX_SUSE',
    'LINUX_ALPINE',
    'MACOS',
    'CHROME_OS',
    'ANDROID',
    'IOS',
    'OUTRO6'
]

df_so = (
    datafram
    .melt(
        id_vars=id_vars,
        value_vars=so_cols,
        var_name="SISTEMAS_OPERACIONAIS",
        value_name="RESPOSTA_SISTEMAS_OPERACIONAIS"
    )
)

df_so["RESPOSTA_SISTEMAS_OPERACIONAIS"] = df_so["RESPOSTA_SISTEMAS_OPERACIONAIS"].replace({'OUTRO6': 'OUTRO'})






so_cols_ = ['SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_SISTEMA_OPERACIONAL_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO']

df_so_ = (
    datafram
    .melt(
        id_vars=id_vars,
        value_vars=so_cols_,
        var_name="SISTEMAS_OPERACIONAIS_APONTADOS",
        value_name="RESPOSTA_SISTEMAS_OPERACIONAIS_APONTADOS"
    )
)






so_cols__ = ['QUAIS_DOS_SEGUINTES_SISTEMAS_OPERACIONAIS_LHE_SAO_PRIORITARIOS'
]

df_so__ = (
    datafram
    .melt(
        id_vars=id_vars,
        value_vars=so_cols__,
        var_name="SISTEMAS_PRIORITARIOS",
        value_name="RESPOSTA_SISTEMAS_PRIORITARIOS"
    )
)





df_final = (
    df_so
    .merge(df_so_, on=["ID","EMAIL","NAME"])
    .merge(df_so__, on=["ID","EMAIL","NAME"])
)


'''
df_final = (
    df_so
    .merge(df_so_.drop(columns=["ID","EMAIL"]), on="NAME")
    .merge(df_so__.drop(columns=["ID","EMAIL"]), on="NAME")
)


df_final = df_so.merge(
    df_so_,
    on="NAME",
    suffixes=("_so", "_prioridade")
)
'''


# Beginning of the studies

In [118]:
#conn.close()

import sqlite3
conn = sqlite3.connect("cepel_data.db")

import pandas as pd
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)

print(tables)


table1 = pd.read_sql_query('''SELECT * FROM Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_0_Identificação1_21''', conn)

table4 = pd.read_sql_query('''SELECT * FROM Composição_do_Stack_Tecnológico_de_Dados_no_Cepel_Parte_3_Interface_DevOps1_18''', conn)

                                                name
0                                          joined_df
1  Composição_do_Stack_Tecnológico_de_Dados_no_Ce...
2  Composição_do_Stack_Tecnológico_de_Dados_no_Ce...
3  Composição_do_Stack_Tecnológico_de_Dados_no_Ce...
4  Composição_do_Stack_Tecnológico_de_Dados_no_Ce...


In [119]:
grupos = {

    "SISTEMA_OPERACIONAL": [
        "WINDOWS_NT_SERVER",
        "WINDOWS_1011",
        "LINUX_DEBIAN_UBUNTU_MINT_KALI",
        "LINUX_ARCH_MANJARO_ENDEAVOUROS",
        "LINUX_RED_HAT_FEDORA_CENTOS_ROCKY",
        "LINUX_SUSE",
        "LINUX_ALPINE",
        "MACOS",
        "CHROME_OS",
        "ANDROID",
        "IOS",
        "OUTRO6"
    ],

    "SISTEMA_OPERACIONAL_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_SISTEMA_OPERACIONAL_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "SISTEMA_OPERACIONAL_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_SISTEMAS_OPERACIONAIS_LHE_SAO_PRIORITARIOS"
    ],

    "VIRTUALIZACAO_E_ACESSO": [
        "SSH",
        "RDP_WINDOWS",
        "VNC_MULTIPLATAFORMA",
        "WSL_WINDOWS_SUBSYSTEM_FOR_LINUX",
        "VMWARE",
        "VIRTUALBOX",
        "PROXMOX",
        "DOCKER",
        "PODMAN",
        "SINGULARITY",
        "DOCKERCOMPOSE",
        "DOCKERSWARM",
        "KUBERNETES",
        "OUTRO7"
    ],

    "VIRTUALIZACAO_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_MODELO_DE_VIRTUALIZACAO_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "VIRTUALIZACAO_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_MODELOS_DE_VIRTUALIZACAO_OU_DE_ACESSO_VIRTUAL_A_RECURSOS_LHE_SAO_PRIORITARIOS"
    ],

    "LINGUAGEM_PROGRAMACAO": [
        "SQL",
        "PYTHON",
        "R",
        "SCALA",
        "JAVA",
        "JAVASCRIPT",
        "TYPESCRIPT",
        "SHELL_SCRIPT",
        "PHP",
        "RUBY",
        "GO",
        "RUST",
        "C_E_C",
        "C",
        "JULIA",
        "FORTRAN",
        "MATLAB",
        "LABVIEW2",
        "OUTRA"
    ],

    "LINGUAGEM_OUTRA": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRA_LINGUAGEM_DE_PROGRAMACAO_VOLTADA_A_DADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "LINGUAGEM_PRIORITARIA": [
        "QUAIS_DAS_SEGUINTES_LINGUAGENS_DE_PROGRAMACAO_VOLTADAS_A_DADOS_LHE_SAO_PRIORITARIAS"
    ],

    "IDES_E_FERRAMENTAS_DEV": [
        "MS_VISUAL_STUDIO",
        "VS_CODE",
        "CURSOR",
        "ANTIGRAVITY",
        "ECLIPSE",
        "INTELLIJ_IDEA",
        "NETBEAMS",
        "PYCHARM",
        "SUBLIME",
        "VIM",
        "EMACS",
        "LABVIEW",
        "CODER_CDE",
        "JUPYTER_LABNOTEBOOK",
        "GITHUB_CODESPACES",
        "KAGGLE_NOTEBOOKS",
        "GOOGLE_COLABORATORY",
        "BACKSTAGE_IDP",
        "OPSLEVEL",
        "COMPASS",
        "PORT",
        "OUTRO10"
    ],

    "IDES_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_DESENVOLVIMENTO_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "IDES_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_DESENVOLVIMENTO_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS"
    ],

    "CONTROLE_DE_VERSAO": [
        "CVS_CONCURRENT_VERSION_SYSTEM",
        "SVN_APACHE_SUBVERSION",
        "MERCURIAL",
        "GIT_GITHUB",
        "GIT_GITLAB",
        "GIT_BITBUCKET",
        "GIT_LFS_LARGE_FILE_STORAGE",
        "DVC_DATA_VERSION_CONTROL",
        "LAKEFS",
        "OUTRO2"
    ],

    "CONTROLE_VERSAO_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_CONTROLE_DE_VERSAO_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "CONTROLE_VERSAO_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_CONTROLE_DE_VERSAO_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS"
    ],

    "INFRAESTRUTURA_COMO_CODIGO": [
        "TERRAFORM",
        "OPENTOFU",
        "PULUMI",
        "CROSSPLANE",
        "ANSIBLE",
        "PUPPET",
        "CHEF",
        "SALT",
        "NORNIR",
        "VAGRANT",
        "OUTRO5"
    ],

    "CI_CD": [
        "JENKINS",
        "TRAVIS_CI",
        "CIRCLE_CI",
        "GITHUB_ACTIONS",
        "GITLAB_CICD",
        "TEAMCITY",
        "BITBUCKET_PIEPELINES",
        "FLUXCD",
        "ARGOCD",
        "OCTOPUSDEPLOY",
        "OUTRO3"
    ],

    "METODOLOGIAS_GESTAO": [
        "PMBOK",
        "SWEBOK",
        "DMBOK",
        "KANBAN",
        "SCRUM",
        "EXTREME_PROGRAMMING_XP",
        "SAFE",
        "OUTRO"
    ],

    "BOAS_PRATICAS_ENGENHARIA": [
        "DEVOPS",
        "DEVSECOPS_OWASP_TOP_10",
        "DATAOPS",
        "MLOPS",
        "CLEAN_ARCHITECTURE_E_CLEAN_CODE",
        "DDD_DOMAINDRIVEN_DESIGN",
        "DESIGN_PATTERNS",
        "BDD_BEHAVIORDRIVEN_DESIGN",
        "TDD_TESTDRIVEN_DEVELOPMENT",
        "CRISPDM",
        "PRINCIPIOS_SOLID",
        "PRINCIPIOS_KISS",
        "PRINCIPIOS_YAGNI",
        "REFACTORING",
        "THE_TWELVEFACTOR_APP",
        "OUTRO9"
    ],

    "FERRAMENTAS_GESTAO_PROJETOS": [
        "SCOPI",
        "BSC_DESIGNER",
        "TRELLO",
        "PLANNER",
        "CLICKUP",
        "JIRA",
        "LINEAR",
        "GLPI",
        "MANTIS",
        "BUGZILLA",
        "OUTRO4"
    ],

    "DOCUMENTACAO_COLABORACAO": [
        "MS_SHAREPOINT",
        "CONFLUENCE",
        "NOTION",
        "MONDAYCOM",
        "MIRO",
        "MS_WHITEBOARD",
        "SLACK",
        "MS_TEAMS",
        "DOXYGEN",
        "DRAWIO",
        "PLANTUML",
        "SWAGGER",
        "CKAN",
        "RESTRUCTUREDTEXT_REST",
        "SPHINX",
        "MKDOCS",
        "READ_THE_DOCS",
        "KEEP_A_CHANGELOG",
        "SCRIBE",
        "OBS_STUDIO",
        "OUTRO8"
    ]
}

import pandas as pd

id_cols = ["ID", "EMAIL", "NAME"]

dfs = []

# grupos.items() -> it will return a view object that displays a list of a dictionary's key-value tuple pairs.
# dict_items([('SISTEMA_OPERACIONAL', ['WINDOWS_NT_SERVER', 'WINDOWS_1011', 'LINUX_DEBIAN_UBUNTU_MINT_KALI', 'LINUX_A....

for categoria, cols in grupos.items():

    df_temp = table4.melt(
        id_vars=id_cols,
        value_vars=cols,
        var_name="TECNOLOGIA",
        value_name="RESPOSTA"
    )

    df_temp["CATEGORIA"] = categoria

    dfs.append(df_temp)
    

In [120]:
df_final = pd.concat(dfs, ignore_index=True)

In [121]:
# Replacing Regex r"OUTRO\d+"
df_final["TECNOLOGIA"] = df_final["TECNOLOGIA"].str.replace(r"OUTRO\d+", "OUTRO", regex=True)

In [122]:
df_final.count()

ID            2970
EMAIL         2970
NAME          2970
TECNOLOGIA    2970
RESPOSTA      2869
CATEGORIA     2970
dtype: int64

In [143]:
grupos = {

    "SISTEMA_OPERACIONAL": [
        "WINDOWS_NT_SERVER",
        "WINDOWS_1011",
        "LINUX_DEBIAN_UBUNTU_MINT_KALI",
        "LINUX_ARCH_MANJARO_ENDEAVOUROS",
        "LINUX_RED_HAT_FEDORA_CENTOS_ROCKY",
        "LINUX_SUSE",
        "LINUX_ALPINE",
        "MACOS",
        "CHROME_OS",
        "ANDROID",
        "IOS",
        "OUTRO6"
    ],

    "SISTEMA_OPERACIONAL_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_SISTEMA_OPERACIONAL_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "SISTEMA_OPERACIONAL_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_SISTEMAS_OPERACIONAIS_LHE_SAO_PRIORITARIOS"
    ],

    "VIRTUALIZACAO_E_ACESSO": [
        "SSH",
        "RDP_WINDOWS",
        "VNC_MULTIPLATAFORMA",
        "WSL_WINDOWS_SUBSYSTEM_FOR_LINUX",
        "VMWARE",
        "VIRTUALBOX",
        "PROXMOX",
        "DOCKER",
        "PODMAN",
        "SINGULARITY",
        "DOCKERCOMPOSE",
        "DOCKERSWARM",
        "KUBERNETES",
        "OUTRO7"
    ],

    "VIRTUALIZACAO_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_MODELO_DE_VIRTUALIZACAO_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "VIRTUALIZACAO_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_MODELOS_DE_VIRTUALIZACAO_OU_DE_ACESSO_VIRTUAL_A_RECURSOS_LHE_SAO_PRIORITARIOS"
    ],

    "LINGUAGEM_PROGRAMACAO": [
        "SQL",
        "PYTHON",
        "R",
        "SCALA",
        "JAVA",
        "JAVASCRIPT",
        "TYPESCRIPT",
        "SHELL_SCRIPT",
        "PHP",
        "RUBY",
        "GO",
        "RUST",
        "C_E_C",
        "C",
        "JULIA",
        "FORTRAN",
        "MATLAB",
        "LABVIEW2",
        "OUTRA"
    ],

    "LINGUAGEM_OUTRA": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRA_LINGUAGEM_DE_PROGRAMACAO_VOLTADA_A_DADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "LINGUAGEM_PRIORITARIA": [
        "QUAIS_DAS_SEGUINTES_LINGUAGENS_DE_PROGRAMACAO_VOLTADAS_A_DADOS_LHE_SAO_PRIORITARIAS"
    ],

    "IDES_E_FERRAMENTAS_DEV": [
        "MS_VISUAL_STUDIO",
        "VS_CODE",
        "CURSOR",
        "ANTIGRAVITY",
        "ECLIPSE",
        "INTELLIJ_IDEA",
        "NETBEAMS",
        "PYCHARM",
        "SUBLIME",
        "VIM",
        "EMACS",
        "LABVIEW",
        "CODER_CDE",
        "JUPYTER_LABNOTEBOOK",
        "GITHUB_CODESPACES",
        "KAGGLE_NOTEBOOKS",
        "GOOGLE_COLABORATORY",
        "BACKSTAGE_IDP",
        "OPSLEVEL",
        "COMPASS",
        "PORT",
        "OUTRO10"
    ],

    "IDES_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_DESENVOLVIMENTO_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "IDES_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_DESENVOLVIMENTO_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS"
    ],

    "CONTROLE_DE_VERSAO": [
        "CVS_CONCURRENT_VERSION_SYSTEM",
        "SVN_APACHE_SUBVERSION",
        "MERCURIAL",
        "GIT_GITHUB",
        "GIT_GITLAB",
        "GIT_BITBUCKET",
        "GIT_LFS_LARGE_FILE_STORAGE",
        "DVC_DATA_VERSION_CONTROL",
        "LAKEFS",
        "OUTRO2"
    ],

    "CONTROLE_VERSAO_OUTRO": [
        "SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_CONTROLE_DE_VERSAO_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO"
    ],

    "CONTROLE_VERSAO_PRIORITARIO": [
        "QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_CONTROLE_DE_VERSAO_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS"
    ],

    "INFRAESTRUTURA_COMO_CODIGO": [
        "TERRAFORM",
        "OPENTOFU",
        "PULUMI",
        "CROSSPLANE",
        "ANSIBLE",
        "PUPPET",
        "CHEF",
        "SALT",
        "NORNIR",
        "VAGRANT",
        "OUTRO5"
    ],

    "CI_CD": [
        "JENKINS",
        "TRAVIS_CI",
        "CIRCLE_CI",
        "GITHUB_ACTIONS",
        "GITLAB_CICD",
        "TEAMCITY",
        "BITBUCKET_PIEPELINES",
        "FLUXCD",
        "ARGOCD",
        "OCTOPUSDEPLOY",
        "OUTRO3"
    ],

    "METODOLOGIAS_GESTAO": [
        "PMBOK",
        "SWEBOK",
        "DMBOK",
        "KANBAN",
        "SCRUM",
        "EXTREME_PROGRAMMING_XP",
        "SAFE",
        "OUTRO"
    ],

    "BOAS_PRATICAS_ENGENHARIA": [
        "DEVOPS",
        "DEVSECOPS_OWASP_TOP_10",
        "DATAOPS",
        "MLOPS",
        "CLEAN_ARCHITECTURE_E_CLEAN_CODE",
        "DDD_DOMAINDRIVEN_DESIGN",
        "DESIGN_PATTERNS",
        "BDD_BEHAVIORDRIVEN_DESIGN",
        "TDD_TESTDRIVEN_DEVELOPMENT",
        "CRISPDM",
        "PRINCIPIOS_SOLID",
        "PRINCIPIOS_KISS",
        "PRINCIPIOS_YAGNI",
        "REFACTORING",
        "THE_TWELVEFACTOR_APP",
        "OUTRO9"
    ],

    "FERRAMENTAS_GESTAO_PROJETOS": [
        "SCOPI",
        "BSC_DESIGNER",
        "TRELLO",
        "PLANNER",
        "CLICKUP",
        "JIRA",
        "LINEAR",
        "GLPI",
        "MANTIS",
        "BUGZILLA",
        "OUTRO4"
    ],

    "DOCUMENTACAO_COLABORACAO": [
        "MS_SHAREPOINT",
        "CONFLUENCE",
        "NOTION",
        "MONDAYCOM",
        "MIRO",
        "MS_WHITEBOARD",
        "SLACK",
        "MS_TEAMS",
        "DOXYGEN",
        "DRAWIO",
        "PLANTUML",
        "SWAGGER",
        "CKAN",
        "RESTRUCTUREDTEXT_REST",
        "SPHINX",
        "MKDOCS",
        "READ_THE_DOCS",
        "KEEP_A_CHANGELOG",
        "SCRIBE",
        "OBS_STUDIO",
        "OUTRO8"
    ]
}


#dfs_resultados = {}
#for categoria in grupos.keys():
    #df_temp = (
        #resultado[resultado["CATEGORIA"] == categoria]
        #.groupby([
            #"EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO",
            #"HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS",
            #"RESPOSTA"
        #])
        #.size()
        #.reset_index(name="TOTAL")
    #)
    #dfs_resultados[categoria] = df_temp


dfs_resultados = {}

for categoria in grupos.keys():

    df_result = (
        resultado[resultado["CATEGORIA"] == categoria]
        .groupby(["EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO","HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS","TECNOLOGIA"])
        .size()
        .reset_index(name="TOTAL")
    )
    df_result

    import pandas as pd

    # Salvando de forma básica
    df_result.to_csv(f'{categoria}.csv', index=False)

    print(f"Arquivo salvo como '{categoria}.csv'")

#res = resultado.groupby(["EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO", "SISTEMAS_OPERACIONAIS", "RESPOSTA"]).size().reset_index(name="quantidade")

Arquivo salvo como 'SISTEMA_OPERACIONAL.csv'
Arquivo salvo como 'SISTEMA_OPERACIONAL_OUTRO.csv'
Arquivo salvo como 'SISTEMA_OPERACIONAL_PRIORITARIO.csv'
Arquivo salvo como 'VIRTUALIZACAO_E_ACESSO.csv'
Arquivo salvo como 'VIRTUALIZACAO_OUTRO.csv'
Arquivo salvo como 'VIRTUALIZACAO_PRIORITARIO.csv'
Arquivo salvo como 'LINGUAGEM_PROGRAMACAO.csv'
Arquivo salvo como 'LINGUAGEM_OUTRA.csv'
Arquivo salvo como 'LINGUAGEM_PRIORITARIA.csv'
Arquivo salvo como 'IDES_E_FERRAMENTAS_DEV.csv'
Arquivo salvo como 'IDES_OUTRO.csv'
Arquivo salvo como 'IDES_PRIORITARIO.csv'
Arquivo salvo como 'CONTROLE_DE_VERSAO.csv'
Arquivo salvo como 'CONTROLE_VERSAO_OUTRO.csv'
Arquivo salvo como 'CONTROLE_VERSAO_PRIORITARIO.csv'
Arquivo salvo como 'INFRAESTRUTURA_COMO_CODIGO.csv'
Arquivo salvo como 'CI_CD.csv'
Arquivo salvo como 'METODOLOGIAS_GESTAO.csv'
Arquivo salvo como 'BOAS_PRATICAS_ENGENHARIA.csv'
Arquivo salvo como 'FERRAMENTAS_GESTAO_PROJETOS.csv'
Arquivo salvo como 'DOCUMENTACAO_COLABORACAO.csv'


In [141]:
df_result

,EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO,HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS,TECNOLOGIA,TOTAL
0,DT/DGA - Depto de Tecnologia em Gestão de Ativos,3-5 anos,CKAN,1
1,DT/DGA - Depto de Tecnologia em Gestão de Ativos,3-5 anos,CONFLUENCE,1
2,DT/DGA - Depto de Tecnologia em Gestão de Ativos,3-5 anos,DOXYGEN,1
3,DT/DGA - Depto de Tecnologia em Gestão de Ativos,3-5 anos,DRAWIO,1
4,DT/DGA - Depto de Tecnologia em Gestão de Ativos,3-5 anos,KEEP_A_CHANGELOG,1
...,...,...,...,...
268,DT/DTS - Depto de Transição Energética e Suste...,5-10 anos,RESTRUCTUREDTEXT_REST,1
269,DT/DTS - Depto de Transição Energética e Suste...,5-10 anos,SCRIBE,1
270,DT/DTS - Depto de Transição Energética e Suste...,5-10 anos,SLACK,1
271,DT/DTS - Depto de Transição Energética e Suste...,5-10 anos,SPHINX,1


```
Solução

dfs_resultados = {}

for categoria in grupos.keys():
    
    df_temp = (
        resultado[resultado["CATEGORIA"] == categoria]
        .groupby([
            "EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO",
            "HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS",
            "RESPOSTA"
        ])
        .size()
        .reset_index(name="TOTAL")
    )
    
    dfs_resultados[categoria] = df_temp

Agora você terá um DataFrame para cada categoria.

Como acessar
dfs_resultados["BOAS_PRATICAS_ENGENHARIA"]

ou

dfs_resultados["SISTEMA_OPERACIONAL"]
Se quiser imprimir todos
for categoria, df in dfs_resultados.items():
    print(f"\n===== {categoria} =====")
    print(df)
Se quiser um único dataframe final com a categoria

Isso geralmente é melhor para análise:

lista_df = []

for categoria in grupos.keys():
    
    df_temp = (
        resultado[resultado["CATEGORIA"] == categoria]
        .groupby([
            "EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO",
            "HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS",
            "RESPOSTA"
        ])
        .size()
        .reset_index(name="TOTAL")
    )
    
    df_temp["CATEGORIA"] = categoria
    lista_df.append(df_temp)

df_final = pd.concat(lista_df, ignore_index=True)
```

In [ ]:
,"TECNOLOGIA"

In [150]:
lista_df = []

for categoria in grupos.keys():
    
    df_temp = (
        resultado[resultado["CATEGORIA"] == categoria]
        .groupby([
            "EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO",
            "HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS",
            "RESPOSTA","TECNOLOGIA"
        ])
        .size()
        .reset_index(name="TOTAL")
    )
    
    df_temp["CATEGORIA"] = categoria
    lista_df.append(df_temp)

df_final = pd.concat(lista_df, ignore_index=True)

In [ ]:
df_final

In [152]:
df_final.to_csv('df_final.csv', index=False)


In [ ]:
dfs_resultados = {}

for categoria in grupos.keys():
    
    df_temp = (
        resultado[resultado["CATEGORIA"] == categoria]
        .groupby([
            "EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO",
            "HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS",
            "RESPOSTA"
        ])
        .size()
        .reset_index(name="TOTAL")
    )
    
    dfs_resultados[categoria] = df_temp


#dfs_resultados["BOAS_PRATICAS_ENGENHARIA"]

#dfs_resultados["SISTEMA_OPERACIONAL"]

#for categoria, df in dfs_resultados.items():
    #print(f"\n===== {categoria} =====")
    #print(df)


lista_df = []

for categoria in grupos.keys():
    
    df_temp = (
        resultado[resultado["CATEGORIA"] == categoria]
        .groupby([
            "EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO",
            "HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS",
            "RESPOSTA"
        ])
        .size()
        .reset_index(name="TOTAL")
    )
    
    df_temp["CATEGORIA"] = categoria
    lista_df.append(df_temp)

df_final = pd.concat(lista_df, ignore_index=True)

In [ ]:
df_final

In [147]:
# Salvando de forma básica
df_final.to_csv('df_final.csv', index=False)

#print(f"Arquivo salvo como '{categoria}.csv'")


In [ ]:
# df_final = df_final.dropna(subset=["RESPOSTA"])

In [ ]:
# df_final = df_final[df_final["RESPOSTA"] != ""]

In [108]:
#df_final.groupby(
#    ["NAME", "CATEGORIA", "TECNOLOGIA", "RESPOSTA"]
#).size().reset_index(name="TOTAL")


#resultado = pd.merge(
#    table1, 
#   df_final , 
#    on=['NAME', 'NAME'],
#    how='left', suffixes=('_table1', '_df_long')
#)

resultado = (
    table1
    .merge(
        df_final.drop(columns=["ID","EMAIL"], errors="ignore"),
        on="NAME",
        how="left"
    )
)

In [ ]:
resultado
#.head()

In [110]:
print(list(resultado.columns))

['ID', 'START_TIME', 'COMPLETION_TIME', 'EMAIL', 'NAME', 'LAST_MODIFIED_TIME', 'FAVOR_INSERIR_SEU_NOME_COMPLETO', 'FAVOR_INSERIR_SEU_EMAIL_CEPELBR', 'EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO', 'FAVOR_APONTAR_SUA_FAIXA_ETARIA', 'FAVOR_APONTAR_O_SEU_TEMPO_DE_CASA_NO_CEPEL', 'FAVOR_APONTAR_SEU_MAIOR_NIVEL_DE_FORMACAO_EDUCACIONAL', 'HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS', 'HA_QUANTO_TEMPO_VOCE_JA_TRABALHA_COM_DADOS_GOVERNANCA_ENGENHARIA_CIENCIA_OU_ANALISE_DE_DADOS', 'TECNOLOGIA', 'RESPOSTA', 'CATEGORIA']


In [55]:
df_final[df_final["CATEGORIA"] == "OUTRO_SISTEMA_OPERACIONAL"] \
.groupby("TECNOLOGIA") \
.size()

TECNOLOGIA
SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_SISTEMA_OPERACIONAL_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO    18
dtype: int64

In [7]:
table4 = table4[['ID',
 'EMAIL',
 'NAME',
 'LAST_MODIFIED_TIME',
 'WINDOWS_NT_SERVER',
 'WINDOWS_1011',
 'LINUX_DEBIAN_UBUNTU_MINT_KALI',
 'LINUX_ARCH_MANJARO_ENDEAVOUROS',
 'LINUX_RED_HAT_FEDORA_CENTOS_ROCKY',
 'LINUX_SUSE',
 'LINUX_ALPINE',
 'MACOS',
 'CHROME_OS',
 'ANDROID',
 'IOS',
 'OUTRO6']]


df_long = table4.melt(
    id_vars=['ID', 'EMAIL', 'NAME'],
    value_vars= ['WINDOWS_NT_SERVER', 'WINDOWS_1011', 'LINUX_DEBIAN_UBUNTU_MINT_KALI', 
                 'LINUX_ARCH_MANJARO_ENDEAVOUROS', 'LINUX_RED_HAT_FEDORA_CENTOS_ROCKY', 
                 'LINUX_SUSE', 'LINUX_ALPINE','MACOS', 'CHROME_OS', 'ANDROID', 'IOS', 'OUTRO6'],
    var_name='SISTEMAS_OPERACIONAIS',
    value_name='RESPOSTA'
)

df_long['SISTEMAS_OPERACIONAIS'] = df_long['SISTEMAS_OPERACIONAIS'].replace({'OUTRO6': 'OUTRO'})

In [1]:
#res = resultado.groupby(["EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO", "SISTEMAS_OPERACIONAIS", "RESPOSTA"]).size().reset_index(name="quantidade")

#res['SISTEMAS'] = (
#    res['SISTEMAS_OPERACIONAIS'].fillna('Sem tecnologia').astype(str)
#    + ' | ' +
#    res['RESPOSTA'].fillna('Sem resposta').astype(str))

In [ ]:
df_long

In [30]:
resultado = pd.merge(
    table1, 
    df_long, 
    on=['NAME', 'NAME'],
    how='left', suffixes=('_table1', '_df_long')
)

In [ ]:
resultado.head(3)

In [35]:
res = resultado.groupby(["EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO", "SISTEMAS_OPERACIONAIS", "RESPOSTA"]).size().reset_index(name="quantidade")

res['SISTEMAS'] = (
    res['SISTEMAS_OPERACIONAIS'].fillna('Sem tecnologia').astype(str)
    + ' | ' +
    res['RESPOSTA'].fillna('Sem resposta').astype(str))

In [ ]:
res

In [37]:
res.to_csv('EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO.csv', index=False)

print("Arquivo salvo como 'dados_melted.csv'")

Arquivo salvo como 'dados_melted.csv'


In [ ]:
resultado = pd.merge(
    table1, 
    table4, 
    on=['NAME', 'NAME'],
    how='left'
)

resultado

In [46]:
import pandas as pd

# Se 'resultado' ainda não existir, crie a partir do merge:
# resultado = pd.merge(table1, table4, on='NAME', how='left')

# Nomes das colunas (ajuste se necessário)
col_depto = 'EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO'
col_tempo_prog = 'HA_QUANTO_TEMPO_VOCE_JA_PROGRAMA_DESENVOLVE_SISTEMAS'
col_obs = 'OBS_STUDIO'

# Verificação básica
for c in [col_depto, col_tempo_prog, col_obs, 'NAME']:
    if c not in resultado.columns:
        raise KeyError(f'Coluna ausente: {c}')

# Defina quais estados contam como "usar OBS Studio"
# Somente quem usa atualmente:
status_validos = ['Uso atualmente']
# Se quiser incluir também "Já usei", use:
# status_validos = ['Uso atualmente', 'Já usei']

filtro_obs = resultado[col_obs].astype(str).str.strip().isin(status_validos)

# Contagem de pessoas que usam OBS (por depto e tempo de programação)
usa_obs = (
    resultado.loc[filtro_obs]
    .groupby([col_depto, col_tempo_prog], dropna=False)['NAME']
    .nunique()  # garante contagem por pessoa
    .reset_index(name='qtd_pessoas_usa_obs')
)

# Total de pessoas por depto e tempo de programação (para percentual)
totais = (
    resultado
    .groupby([col_depto, col_tempo_prog], dropna=False)['NAME']
    .nunique()
    .reset_index(name='total_pessoas')
)

# Junta e calcula percentual
resumo = usa_obs.merge(totais, on=[col_depto, col_tempo_prog], how='right')
resumo['qtd_pessoas_usa_obs'] = resumo['qtd_pessoas_usa_obs'].fillna(0).astype(int)
resumo['pct_usa_obs'] = (resumo['qtd_pessoas_usa_obs'] / resumo['total_pessoas'] * 100).round(2)

# Ordena para facilitar leitura
resumo = resumo.sort_values([col_depto, col_tempo_prog])

In [1]:
'''
import pandas as pd

# Versão corrigida - GROUP BY com múltiplas colunas
resumo_por_departamento = resultado.groupby([
    'EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO',
    'FAVOR_APONTAR_SUA_FAIXA_ETARIA'  # Adicionei aspas e vírgula
]).agg({
    'NAME': 'count',  # Conta quantas pessoas por departamento, faixa etária e tecnologia
    'resposta': lambda x: x.notna().sum()  # Conta respostas não nulas
    # Removido 'tecnologia': 'count' porque tecnologia agora é uma das colunas de agrupamento
}).rename(columns={
    'NAME': 'total_mencoes',
    'resposta': 'respostas_validas'
})


resumo_por_departamento
'''

"\nimport pandas as pd\n\n# Versão corrigida - GROUP BY com múltiplas colunas\nresumo_por_departamento = resultado.groupby([\n    'EM_QUAL_DEPARTAMENTO_DO_CEPEL_VOCE_ESTA_ALOCADO',\n    'FAVOR_APONTAR_SUA_FAIXA_ETARIA'  # Adicionei aspas e vírgula\n]).agg({\n    'NAME': 'count',  # Conta quantas pessoas por departamento, faixa etária e tecnologia\n    'resposta': lambda x: x.notna().sum()  # Conta respostas não nulas\n    # Removido 'tecnologia': 'count' porque tecnologia agora é uma das colunas de agrupamento\n}).rename(columns={\n    'NAME': 'total_mencoes',\n    'resposta': 'respostas_validas'\n})\n\n\nresumo_por_departamento\n"

In [33]:
resultado.to_csv('resultado.csv', index=False)

print("Arquivo salvo como 'dados_melted.csv'")

Arquivo salvo como 'dados_melted.csv'


In [13]:
print(list(table.columns))

['ID', 'START_TIME', 'COMPLETION_TIME', 'EMAIL', 'NAME', 'LAST_MODIFIED_TIME', 'WINDOWS_NT_SERVER', 'WINDOWS_1011', 'LINUX_DEBIAN_UBUNTU_MINT_KALI', 'LINUX_ARCH_MANJARO_ENDEAVOUROS', 'LINUX_RED_HAT_FEDORA_CENTOS_ROCKY', 'LINUX_SUSE', 'LINUX_ALPINE', 'MACOS', 'CHROME_OS', 'ANDROID', 'IOS', 'OUTRO6', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_SISTEMA_OPERACIONAL_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO', 'QUAIS_DOS_SEGUINTES_SISTEMAS_OPERACIONAIS_LHE_SAO_PRIORITARIOS', 'SSH', 'RDP_WINDOWS', 'VNC_MULTIPLATAFORMA', 'WSL_WINDOWS_SUBSYSTEM_FOR_LINUX', 'VMWARE', 'VIRTUALBOX', 'PROXMOX', 'DOCKER', 'PODMAN', 'SINGULARITY', 'DOCKERCOMPOSE', 'DOCKERSWARM', 'KUBERNETES', 'OUTRO7', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_MODELO_DE_VIRTUALIZACAO_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO', 'QUAIS_DOS_SEGUINTES_MODELOS_DE_VIRTUALIZACAO_OU_DE_ACESSO_VIRTUAL_A_RECURSOS_LHE_SAO_PRIORITARIOS', 'SQL', 'PYTHON', 'R', 'SCALA', 'JAVA', 'JAVASCRIPT', 'TYPESCRIPT', 'SHE

In [ ]:
df_long = table.melt(
    id_vars=['NAME'],           # Colunas que permanecem fixas
    value_vars=['WINDOWS_NT_SERVER', 'WINDOWS_1011', 'LINUX_DEBIAN_UBUNTU_MINT_KALI', 'LINUX_ARCH_MANJARO_ENDEAVOUROS', 'LINUX_RED_HAT_FEDORA_CENTOS_ROCKY', 'LINUX_SUSE', 'LINUX_ALPINE', 'MACOS', 'CHROME_OS', 'ANDROID', 'IOS', 'OUTRO6', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_SISTEMA_OPERACIONAL_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO', 'QUAIS_DOS_SEGUINTES_SISTEMAS_OPERACIONAIS_LHE_SAO_PRIORITARIOS', 'SSH', 'RDP_WINDOWS', 'VNC_MULTIPLATAFORMA', 'WSL_WINDOWS_SUBSYSTEM_FOR_LINUX', 'VMWARE', 'VIRTUALBOX', 'PROXMOX', 'DOCKER', 'PODMAN', 'SINGULARITY', 'DOCKERCOMPOSE', 'DOCKERSWARM', 'KUBERNETES', 'OUTRO7', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_MODELO_DE_VIRTUALIZACAO_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO', 'QUAIS_DOS_SEGUINTES_MODELOS_DE_VIRTUALIZACAO_OU_DE_ACESSO_VIRTUAL_A_RECURSOS_LHE_SAO_PRIORITARIOS', 'SQL', 'PYTHON', 'R', 'SCALA', 'JAVA', 'JAVASCRIPT', 'TYPESCRIPT', 'SHELL_SCRIPT', 'PHP', 'RUBY', 'GO', 'RUST', 'C_E_C', 'C', 'JULIA', 'FORTRAN', 'MATLAB', 'LABVIEW2', 'OUTRA', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRA_LINGUAGEM_DE_PROGRAMACAO_VOLTADA_A_DADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO', 'QUAIS_DAS_SEGUINTES_LINGUAGENS_DE_PROGRAMACAO_VOLTADAS_A_DADOS_LHE_SAO_PRIORITARIAS', 'MS_VISUAL_STUDIO', 'VS_CODE', 'CURSOR', 'ANTIGRAVITY', 'ECLIPSE', 'INTELLIJ_IDEA', 'NETBEAMS', 'PYCHARM', 'SUBLIME', 'VIM', 'EMACS', 'LABVIEW', 'CODER_CDE', 'JUPYTER_LABNOTEBOOK', 'GITHUB_CODESPACES', 'KAGGLE_NOTEBOOKS', 'GOOGLE_COLABORATORY', 'BACKSTAGE_IDP', 'OPSLEVEL', 'COMPASS', 'PORT', 'OUTRO10', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_DESENVOLVIMENTO_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO', 'QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_DESENVOLVIMENTO_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS', 'CVS_CONCURRENT_VERSION_SYSTEM', 'SVN_APACHE_SUBVERSION', 'MERCURIAL', 'GIT_GITHUB', 'GIT_GITLAB', 'GIT_BITBUCKET', 'GIT_LFS_LARGE_FILE_STORAGE', 'DVC_DATA_VERSION_CONTROL', 'LAKEFS', 'OUTRO2', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_CONTROLE_DE_VERSAO_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO', 'QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_CONTROLE_DE_VERSAO_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS', 'TERRAFORM', 'OPENTOFU', 'PULUMI', 'CROSSPLANE', 'ANSIBLE', 'PUPPET', 'CHEF', 'SALT', 'NORNIR', 'VAGRANT', 'OUTRO5', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_CONFIGURACAO_DE_RECURSOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO', 'QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_CONFIGURACAO_DE_RECURSOS_LHE_SAO_PRIORITARIOS', 'JENKINS', 'TRAVIS_CI', 'CIRCLE_CI', 'GITHUB_ACTIONS', 'GITLAB_CICD', 'TEAMCITY', 'BITBUCKET_PIEPELINES', 'FLUXCD', 'ARGOCD', 'OCTOPUSDEPLOY', 'OUTRO3', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_CICD_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO', 'QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_CICD_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS', 'PMBOK', 'SWEBOK', 'DMBOK', 'KANBAN', 'SCRUM', 'EXTREME_PROGRAMMING_XP', 'SAFE', 'OUTRO', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_COMPENDIO_DE_BOAS_PRATICAS_PARA_GESTAO_DE_PROJETOS_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO', 'QUAIS_DOS_SEGUINTES_COMPENDIOS_DE_BOAS_PRATICAS_PARA_GESTAO_DE_PROJETOS_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS', 'DEVOPS', 'DEVSECOPS_OWASP_TOP_10', 'DATAOPS', 'MLOPS', 'CLEAN_ARCHITECTURE_E_CLEAN_CODE', 'DDD_DOMAINDRIVEN_DESIGN', 'DESIGN_PATTERNS', 'BDD_BEHAVIORDRIVEN_DESIGN', 'TDD_TESTDRIVEN_DEVELOPMENT', 'CRISPDM', 'PRINCIPIOS_SOLID', 'PRINCIPIOS_KISS', 'PRINCIPIOS_YAGNI', 'REFACTORING', 'THE_TWELVEFACTOR_APP', 'OUTRO9', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_COMPENDIO_DE_BOAS_PRATICAS_EM_DESENVOLVIMENTO_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO', 'QUAIS_DOS_SEGUINTES_COMPENDIOS_DE_BOAS_PRATICAS_EM_DESENVOLVIMENTO_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS', 'SCOPI', 'BSC_DESIGNER', 'TRELLO', 'PLANNER', 'CLICKUP', 'JIRA', 'LINEAR', 'GLPI', 'MANTIS', 'BUGZILLA', 'OUTRO4', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_APOIO_METODOLOGICO_AO_DESENVOLVIMENTO_DE_SOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO', 'QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_APOIO_METODOLOGICO_AO_DESENVOLVIMENTO_DE_SOFTWAREDADOS_LHE_SAO_PRIORITARIOS', 'MS_SHAREPOINT', 'CONFLUENCE', 'NOTION', 'MONDAYCOM', 'MIRO', 'MS_WHITEBOARD', 'SLACK', 'MS_TEAMS', 'DOXYGEN', 'DRAWIO', 'PLANTUML', 'SWAGGER', 'CKAN', 'RESTRUCTUREDTEXT_REST', 'SPHINX', 'MKDOCS', 'READ_THE_DOCS', 'KEEP_A_CHANGELOG', 'SCRIBE', 'OBS_STUDIO', 'OUTRO8', 'SE_VOCE_APONTOU_TER_USADO_USAR_ATUALMENTE_OU_QUERER_USAR_OUTRO_FRAMEWORKFERRAMENTA_DE_DOCUMENTACAO_DE_PROJETOSSOFTWAREDADOS_ACIMA_POR_FAVOR_IDENTIFIQUE_QUAL_ABAIXO', 'QUAIS_DOS_SEGUINTES_FRAMEWORKSFERRAMENTAS_DE_DOCUMENTACAO_DE_PROJETOSSOFTWAREDADOS_LHE_SAO_PRIORITARIOS'],  # Colunas a serem "derretidas"
    var_name='mes',             # Nome da nova coluna de variáveis
    value_name='vendas',
    value_name='v'         # Nome da nova coluna de valores
)

In [19]:
import pandas as pd

# Salvando de forma básica
df_long.to_csv('dados_melted.csv', index=False)

print("Arquivo salvo como 'dados_melted.csv'")

Arquivo salvo como 'dados_melted.csv'
